In [20]:
import pandas as pd
import re
import numpy as np
import torch
import random
from itables import show
import pubchempy as pcp
import rdkit
from rdkit import Chem
from rdkit.Chem import Descriptors
from descriptastorus.descriptors import rdDescriptors
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from functools import lru_cache
import xgboost
from sklearn.model_selection import train_test_split

In [21]:
# Set random seed for reproducibility
torch.manual_seed(101)
np.random.seed(101)
random.seed(101)

In [22]:
df = pd.read_csv('Cleaned_VLE_Data_with_Smiles_and_Mols_and_Descriptors.csv',index_col = 0)
print(df.shape)
df.head()

(106383, 412)


,property,value,phase,"Temperature, K","Pressure, kPa",Mole fraction,Component 1,Component 2,Smiles 1,Smiles 2,...,"('fr_sulfonamd', <class 'numpy.float64'>).1","('fr_sulfone', <class 'numpy.float64'>).1","('fr_term_acetylene', <class 'numpy.float64'>).1","('fr_tetrazole', <class 'numpy.float64'>).1","('fr_thiazole', <class 'numpy.float64'>).1","('fr_thiocyan', <class 'numpy.float64'>).1","('fr_thiophene', <class 'numpy.float64'>).1","('fr_unbrch_alkane', <class 'numpy.float64'>).1","('fr_urea', <class 'numpy.float64'>).1","('qed', <class 'numpy.float64'>).1"
0,"Thermal conductivity, W/m/K",0.02096,Gas,300.0,724.0,0.2493,carbon dioxide,methane,C(=O)=O,C,...,1.593061e-17,5.766101e-14,2.957989e-11,0.168378,0.16738,1.481515e-18,2.324150e-16,4.703598e-08,0.166633,0.187493
1,"Thermal conductivity, W/m/K",0.02125,Gas,300.0,1288.0,0.2493,carbon dioxide,methane,C(=O)=O,C,...,1.593061e-17,5.766101e-14,2.957989e-11,0.168378,0.16738,1.481515e-18,2.324150e-16,4.703598e-08,0.166633,0.187493
2,"Thermal conductivity, W/m/K",0.02161,Gas,300.0,1835.0,0.2493,carbon dioxide,methane,C(=O)=O,C,...,1.593061e-17,5.766101e-14,2.957989e-11,0.168378,0.16738,1.481515e-18,2.324150e-16,4.703598e-08,0.166633,0.187493
3,"Thermal conductivity, W/m/K",0.02197,Gas,300.0,2335.0,0.2493,carbon dioxide,methane,C(=O)=O,C,...,1.593061e-17,5.766101e-14,2.957989e-11,0.168378,0.16738,1.481515e-18,2.324150e-16,4.703598e-08,0.166633,0.187493
4,"Thermal conductivity, W/m/K",0.02245,Gas,300.0,2820.0,0.2493,carbon dioxide,methane,C(=O)=O,C,...,1.593061e-17,5.766101e-14,2.957989e-11,0.168378,0.16738,1.481515e-18,2.324150e-16,4.703598e-08,0.166633,0.187493


In [23]:
# --- Implementation: Create the 'strat_key' ---
# We convert each pair of components to a tuple.
df['strat_key'] = df.apply(lambda row: (row['Component 1'], row['Component 2']), axis=1)

print("DataFrame with the new 'strat_key' column:")
df.head()

DataFrame with the new 'strat_key' column:


C:\Users\harry\AppData\Local\Temp\ipykernel_24076\3099795945.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['strat_key'] = df.apply(lambda row: (row['Component 1'], row['Component 2']), axis=1)


,property,value,phase,"Temperature, K","Pressure, kPa",Mole fraction,Component 1,Component 2,Smiles 1,Smiles 2,...,"('fr_sulfone', <class 'numpy.float64'>).1","('fr_term_acetylene', <class 'numpy.float64'>).1","('fr_tetrazole', <class 'numpy.float64'>).1","('fr_thiazole', <class 'numpy.float64'>).1","('fr_thiocyan', <class 'numpy.float64'>).1","('fr_thiophene', <class 'numpy.float64'>).1","('fr_unbrch_alkane', <class 'numpy.float64'>).1","('fr_urea', <class 'numpy.float64'>).1","('qed', <class 'numpy.float64'>).1",strat_key
0,"Thermal conductivity, W/m/K",0.02096,Gas,300.0,724.0,0.2493,carbon dioxide,methane,C(=O)=O,C,...,5.766101e-14,2.957989e-11,0.168378,0.16738,1.481515e-18,2.324150e-16,4.703598e-08,0.166633,0.187493,"(carbon dioxide, methane)"
1,"Thermal conductivity, W/m/K",0.02125,Gas,300.0,1288.0,0.2493,carbon dioxide,methane,C(=O)=O,C,...,5.766101e-14,2.957989e-11,0.168378,0.16738,1.481515e-18,2.324150e-16,4.703598e-08,0.166633,0.187493,"(carbon dioxide, methane)"
2,"Thermal conductivity, W/m/K",0.02161,Gas,300.0,1835.0,0.2493,carbon dioxide,methane,C(=O)=O,C,...,5.766101e-14,2.957989e-11,0.168378,0.16738,1.481515e-18,2.324150e-16,4.703598e-08,0.166633,0.187493,"(carbon dioxide, methane)"
3,"Thermal conductivity, W/m/K",0.02197,Gas,300.0,2335.0,0.2493,carbon dioxide,methane,C(=O)=O,C,...,5.766101e-14,2.957989e-11,0.168378,0.16738,1.481515e-18,2.324150e-16,4.703598e-08,0.166633,0.187493,"(carbon dioxide, methane)"
4,"Thermal conductivity, W/m/K",0.02245,Gas,300.0,2820.0,0.2493,carbon dioxide,methane,C(=O)=O,C,...,5.766101e-14,2.957989e-11,0.168378,0.16738,1.481515e-18,2.324150e-16,4.703598e-08,0.166633,0.187493,"(carbon dioxide, methane)"


In [24]:
# Use .value_counts() to get the raw count of each unique key
combination_counts = df['strat_key'].value_counts()

print("\n--- Absolute Counts of Each Combination ---")
print(combination_counts)
print(f"\nMean count per combination: {combination_counts.mean()}")
print(f"\nMean count per combination: {combination_counts.median()}")



--- Absolute Counts of Each Combination ---
strat_key
(ethanol, heptane)                                                         1914
(methanol, water)                                                          1849
(acetone, heptane)                                                         1747
(dimethyl carbonate, hexane)                                               1440
(ethanol, water)                                                           1349
                                                                           ... 
((s)-2-aminopropanoic acid, water)                                            1
(l-serine, water)                                                             1
(l-threonine, water)                                                          1
(1-butyl-3-methylimidazolium tetrafluoroborate, nitromethane)                 1
(1-butyl-3-methylimidazolium tetrafluoroborate, n,n-dimethylethanamide)       1
Name: count, Length: 470, dtype: int64

Mean count per combinatio

In [25]:
df['combination_counts'] = df['strat_key'].map(df['strat_key'].value_counts())

C:\Users\harry\AppData\Local\Temp\ipykernel_24076\1381006066.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['combination_counts'] = df['strat_key'].map(df['strat_key'].value_counts())


In [34]:
print(df.shape)
df.head()

(106383, 414)


,property,value,phase,"Temperature, K","Pressure, kPa",Mole fraction,Component 1,Component 2,Smiles 1,Smiles 2,...,"('fr_term_acetylene', <class 'numpy.float64'>).1","('fr_tetrazole', <class 'numpy.float64'>).1","('fr_thiazole', <class 'numpy.float64'>).1","('fr_thiocyan', <class 'numpy.float64'>).1","('fr_thiophene', <class 'numpy.float64'>).1","('fr_unbrch_alkane', <class 'numpy.float64'>).1","('fr_urea', <class 'numpy.float64'>).1","('qed', <class 'numpy.float64'>).1",strat_key,combination_counts
0,"Thermal conductivity, W/m/K",0.02096,Gas,300.0,724.0,0.2493,carbon dioxide,methane,C(=O)=O,C,...,2.957989e-11,0.168378,0.16738,1.481515e-18,2.324150e-16,4.703598e-08,0.166633,0.187493,"(carbon dioxide, methane)",349
1,"Thermal conductivity, W/m/K",0.02125,Gas,300.0,1288.0,0.2493,carbon dioxide,methane,C(=O)=O,C,...,2.957989e-11,0.168378,0.16738,1.481515e-18,2.324150e-16,4.703598e-08,0.166633,0.187493,"(carbon dioxide, methane)",349
2,"Thermal conductivity, W/m/K",0.02161,Gas,300.0,1835.0,0.2493,carbon dioxide,methane,C(=O)=O,C,...,2.957989e-11,0.168378,0.16738,1.481515e-18,2.324150e-16,4.703598e-08,0.166633,0.187493,"(carbon dioxide, methane)",349
3,"Thermal conductivity, W/m/K",0.02197,Gas,300.0,2335.0,0.2493,carbon dioxide,methane,C(=O)=O,C,...,2.957989e-11,0.168378,0.16738,1.481515e-18,2.324150e-16,4.703598e-08,0.166633,0.187493,"(carbon dioxide, methane)",349
4,"Thermal conductivity, W/m/K",0.02245,Gas,300.0,2820.0,0.2493,carbon dioxide,methane,C(=O)=O,C,...,2.957989e-11,0.168378,0.16738,1.481515e-18,2.324150e-16,4.703598e-08,0.166633,0.187493,"(carbon dioxide, methane)",349


In [35]:
data_size = len(df)
print("--- Original Dataset Info ---")
print(f"Total rows: {len(df)}")
print("\n")


# --- 2. Create the Stratification Column ---
# This is the core of the custom stratification logic.
# We create a new column that will serve as the 'y' for stratification.
df['strata_group'] = np.where(df['combination_counts'] > 50, 'high_count', 'low_count')

print("--- Stratification Group Counts (Original) ---")
print(df['strata_group'].value_counts(normalize=True))
print("\n")


# --- 3. First Split: 80% Train, 20% Temp (for Val/Test) ---
# We split the features (X) and the stratification key (y)
x = df
y = df['strata_group']

# test_size=0.2 creates an 80/20 split.
# stratify=y ensures both train and temp sets have the same proportion of strata_group as the original df.
x_train, x_temp, y_train, y_temp = train_test_split(
    x, y, test_size=0.2, random_state=101, stratify=y
)


# --- 4. Second Split: 10% Validation, 10% Test ---
# We split the 20% temp set into two 10% sets (50% of the temp set).
# We stratify again on the y_temp from the previous split.
x_val, x_test, y_val, y_test = train_test_split(
    x_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)


# --- 5. Finalize and Clean Up DataFrames ---
# The split is done. Let's assign them to clean variable names.
train_df = x_train.copy()
val_df = x_val.copy()
test_df = x_test.copy()

# It's good practice to drop the temporary stratification column
# if it's not a feature you intend to use in your model.
train_df.drop(columns=['strata_group'], inplace=True)
val_df.drop(columns=['strata_group'], inplace=True)
test_df.drop(columns=['strata_group'], inplace=True)


# --- 6. Verification ---
# Let's verify the sizes and the stratification distribution.
print("--- Final Set Sizes ---")
print(f"Training set size: {len(train_df)} ({len(train_df)/len(df):.0%})")
print(f"Validation set size: {len(val_df)} ({len(val_df)/len(df):.0%})")
print(f"Test set size: {len(test_df)} ({len(test_df)/len(df):.0%})")
print("\n")

print("--- Stratification Verification ---")
# We need to re-create the strata_group on the final sets to check the distribution
print("Original distribution:\n", df['strata_group'].value_counts(normalize=True), "\n")
print("Training set distribution:\n", train_df.assign(strata_group=np.where(train_df['combination_counts'] > 50, 'high_count', 'low_count'))['strata_group'].value_counts(normalize=True), "\n")
print("Validation set distribution:\n", val_df.assign(strata_group=np.where(val_df['combination_counts'] > 50, 'high_count', 'low_count'))['strata_group'].value_counts(normalize=True), "\n")
print("Test set distribution:\n", test_df.assign(strata_group=np.where(test_df['combination_counts'] > 50, 'high_count', 'low_count'))['strata_group'].value_counts(normalize=True), "\n")


--- Original Dataset Info ---
Total rows: 106383


--- Stratification Group Counts (Original) ---
strata_group
high_count    0.992837
low_count     0.007163
Name: proportion, dtype: float64




C:\Users\harry\AppData\Local\Temp\ipykernel_24076\3443697994.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['strata_group'] = np.where(df['combination_counts'] > 50, 'high_count', 'low_count')


--- Final Set Sizes ---
Training set size: 85106 (80%)
Validation set size: 10638 (10%)
Test set size: 10639 (10%)


--- Stratification Verification ---
Original distribution:
 strata_group
high_count    0.992837
low_count     0.007163
Name: proportion, dtype: float64 

Training set distribution:
 strata_group
high_count    0.992832
low_count     0.007168
Name: proportion, dtype: float64 

Validation set distribution:
 strata_group
high_count    0.992856
low_count     0.007144
Name: proportion, dtype: float64 

Test set distribution:
 strata_group
high_count    0.992856
low_count     0.007144
Name: proportion, dtype: float64 

